# RECAST Baseline Evaluation Driver (Colab)

**Google Colab notebook for evaluating any HuggingFace model on RECAST-30K.**

---

### Quick Start
1. Clone the repo and upload it to Google Drive
2. Add a `.env` file to the **project root** on Drive with `HF_TOKEN=hf_...` (required for gated models like Llama)
3. Set your **model name** and **Drive path** in **Section 0** below
4. **Runtime > Run All** — results CSV + plots land in `output/`

### Expected File Structure (on Google Drive)
```
MyDrive/<project_dir>/
├── .env                       # HF_TOKEN=hf_your_token_here
├── datasets/
│   └── RECAST-30K.json        # 29,939 prompts with constraints
├── baseline_testing/
│   ├── constraint_checker.py  # Deterministic constraint checks (18 types)
│   ├── evaluator.py           # Scoring helpers (per-constraint CSR, hard CSR)
│   └── driver.ipynb           # <-- You are here
└── output/                    # Created automatically
    ├── baseline_<model>_<timestamp>.csv
    └── baseline_<model>_<timestamp>_meta.json
```

### Runtime Notes
| Runtime | Quantization | Batch Size | Full Dataset (~30K) |
|---------|-------------|------------|---------------------|
| **Colab T4** | 4-bit (`QUANT_BITS = 4`) | 4 | ~8–12 hrs |
| **Colab T4** | 8-bit (`QUANT_BITS = 8`) | 4 | ~10–14 hrs |
| **Colab T4** | None (bfloat16) | 4 | May OOM on 7B+ models |

> **Tip:** Set `NUM_SAMPLES = 1000` for a quick stratified evaluation (~1–2 hrs on T4).

---
## Section 0: Configuration

Set your model, Drive path, and run parameters here. Everything else is automatic.

| Parameter | Description |
|-----------|-------------|
| `MODEL_NAME` | Any HuggingFace causal LM (e.g. `Qwen/Qwen2.5-1.5B-Instruct`, `meta-llama/Llama-3.2-1B-Instruct`) |
| `GDRIVE_PROJECT_DIR` | Path to the repo folder on Google Drive (relative to `/content/drive/`) |
| `QUANT_BITS` | `4` or `8` for quantized loading via `bitsandbytes`, `None` for bfloat16 |
| `MAX_NEW_TOKENS` | Max tokens the model generates per prompt |
| `BATCH_SIZE` | Prompts per forward pass. `4` works well on T4 with 4-bit quant |
| `NUM_SAMPLES` | Stratified subsample size. `0` = full dataset (29,939). Use `500`–`1000` for quick runs |

In [ ]:
#@title Configuration { display-mode: "form" }
#@markdown ### Required
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  #@param {type:"string"}
GDRIVE_PROJECT_DIR = "MyDrive/recast_eval"  #@param {type:"string"}

#@markdown ### Optional
HF_TOKEN = ""  #@param {type:"string"}
QUANT_BITS = 4  #@param [4, 8, "None"] {type:"raw"}
MAX_NEW_TOKENS = 512  #@param {type:"integer"}
BATCH_SIZE = 4  #@param {type:"integer"}
NUM_SAMPLES = 0  #@param {type:"integer"}

# Derived paths
DRIVE_BASE = f"/content/drive/{GDRIVE_PROJECT_DIR}"
DATA_DIR = f"{DRIVE_BASE}/datasets"
OUTPUT_DIR = f"{DRIVE_BASE}/output"
DATASET_PATH = f"{DATA_DIR}/RECAST-30K.json"

print(f"Model:       {MODEL_NAME}")
print(f"Project dir: {DRIVE_BASE}")
print(f"Data dir:    {DATA_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Dataset:     {DATASET_PATH}")
print(f"Quantization: {f'{QUANT_BITS}-bit' if QUANT_BITS else 'none (bfloat16)'}")
print(f"Max tokens:  {MAX_NEW_TOKENS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Num samples: {NUM_SAMPLES if NUM_SAMPLES > 0 else 'all'}")

---
## Section 1: Environment Setup

Installs dependencies, mounts Google Drive, validates that all required files exist (`constraint_checker.py`, `evaluator.py`, `RECAST-30K.json`), creates `output/`, and loads your HuggingFace token from `.env` if present.

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes torch tqdm pandas matplotlib seaborn huggingface_hub

In [ ]:
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory ready.")

# Load .env from project root if it exists
env_path = os.path.join(DRIVE_BASE, ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                os.environ[key.strip()] = val.strip()
    print(f"Loaded .env from {env_path}")

# Use .env token if HF_TOKEN wasn't set in the form
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("Using HF_TOKEN from .env")

# Add baseline_testing/ to Python path so evaluator & constraint_checker are importable
BASELINE_DIR = f"{DRIVE_BASE}/baseline_testing"
if BASELINE_DIR not in sys.path:
    sys.path.insert(0, BASELINE_DIR)

# Verify helper modules exist
for fname in ["constraint_checker.py", "evaluator.py"]:
    fpath = os.path.join(BASELINE_DIR, fname)
    assert os.path.exists(fpath), (
        f"{fname} not found at {fpath}. "
        f"Make sure the full repo is uploaded to {GDRIVE_PROJECT_DIR}/"
    )
    print(f"Found {fname}")

# Verify dataset exists
assert os.path.exists(DATASET_PATH), (
    f"Dataset not found at {DATASET_PATH}. "
    f"Make sure the full repo is uploaded to {GDRIVE_PROJECT_DIR}/"
)

---
## Section 2: Load Model

Downloads and loads the HuggingFace model specified in Section 0.

**How the loading logic works (Colab / CUDA):**
- **`QUANT_BITS = 4`** → 4-bit quantization via `bitsandbytes` (default, fits 7B models on T4 16GB)
- **`QUANT_BITS = 8`** → 8-bit quantization via `bitsandbytes` (slightly more accurate, uses more VRAM)
- **`QUANT_BITS = None`** → full bfloat16 (best quality, may OOM on larger models)

> **Tip:** 4-bit is the sweet spot for T4 GPUs. Use 8-bit if you have an A100 or want slightly better quality.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")

print(f"Downloading and loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

if QUANT_BITS == 4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
elif QUANT_BITS == 8:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

model.eval()
print(f"Model loaded: {MODEL_NAME}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## Section 3: Load & Parse Dataset

Loads `RECAST-30K.json` and normalizes each record into a standard format:
- **`prompt`** — the input text sent to the model
- **`constraints`** — list of constraint dicts (each has `type`, `value`, etc.)
- **`difficulty_level`** — `L1` through `L4`, inferred from constraint count if not in the data
- **`reference_response`** — gold response (if available)

If `NUM_SAMPLES > 0`, performs **stratified sampling** (equal samples per difficulty level, seeded with `random.seed(42)` for reproducibility).

In [ ]:
import json
import re
import random
from collections import Counter, defaultdict

# --- Parsing helpers ---

def infer_difficulty(num_constraints):
    if num_constraints <= 2:
        return "L1"
    elif num_constraints <= 4:
        return "L2"
    elif num_constraints <= 7:
        return "L3"
    else:
        return "L4"

def parse_record(record, idx):
    prompt = (record.get("winner_prompt")
              or record.get("prompt")
              or record.get("instruction")
              or "")

    constraints = record.get("constraints", record.get("constraint_list", None))
    if constraints is None:
        constraints = []
    if isinstance(constraints, str):
        try:
            constraints = json.loads(constraints)
        except json.JSONDecodeError:
            constraints = [{"type": "raw", "description": constraints}]
    if not isinstance(constraints, list):
        constraints = [constraints] if constraints else []

    difficulty = (record.get("difficulty_level")
                  or record.get("difficulty")
                  or record.get("level"))
    if difficulty is None:
        difficulty = infer_difficulty(len(constraints))
    difficulty = str(difficulty)
    if not difficulty.startswith("L"):
        difficulty = f"L{difficulty}"

    reference = (record.get("winner_response")
                 or record.get("response")
                 or record.get("output")
                 or "")

    return {
        "id": record.get("id", idx),
        "prompt": prompt,
        "constraints": constraints,
        "num_constraints": len(constraints),
        "difficulty_level": difficulty,
        "reference_response": reference,
    }

# --- Load ---

raw_data = []
if DATASET_PATH.endswith(".jsonl"):
    with open(DATASET_PATH, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                raw_data.append(json.loads(line))
else:
    with open(DATASET_PATH, "r") as f:
        loaded = json.load(f)
        if isinstance(loaded, list):
            raw_data = loaded
        elif isinstance(loaded, dict):
            for key in ["data", "instances", "examples", "records"]:
                if key in loaded and isinstance(loaded[key], list):
                    raw_data = loaded[key]
                    break
            if not raw_data:
                raw_data = [loaded]

dataset = [parse_record(r, i) for i, r in enumerate(raw_data)]

# Stratified sampling if requested
if NUM_SAMPLES > 0:
    by_level = defaultdict(list)
    for item in dataset:
        by_level[item["difficulty_level"]].append(item)
    per_level = max(1, NUM_SAMPLES // len(by_level))
    sampled = []
    for level in sorted(by_level.keys()):
        items = by_level[level]
        random.seed(42)
        sampled.extend(random.sample(items, min(per_level, len(items))))
    dataset = sampled
    print(f"Stratified sampled to {len(dataset)} instances")

level_counts = Counter(d["difficulty_level"] for d in dataset)
print(f"Total instances: {len(dataset)}")
for level in sorted(level_counts.keys()):
    print(f"  {level}: {level_counts[level]}")

---
## Section 4: Inference

Runs zero-shot generation on every prompt using greedy decoding (`do_sample=False`).

**Features:**
- **Checkpointing** — results are written to `output/baseline_<model>_responses.jsonl` every 50 batches. If the runtime disconnects, re-running this cell resumes from where it left off (saved to Drive).
- **Chat template** — automatically applies the model's chat template if available, otherwise uses a plain `User: ... \nAssistant:` format.

> **Runtime estimates (Qwen 1.5B on Colab T4, 4-bit, batch_size=4):**
> | Samples | Approx. Time |
> |---------|-------------|
> | 1,000 | ~1–2 hrs |
> | 10,000 | ~4–5 hrs |
> | 29,939 (all) | ~8–12 hrs |

In [ ]:
import time
from tqdm.auto import tqdm

MODEL_NAME_SAFE = MODEL_NAME.replace("/", "_").replace("-", "_")
checkpoint_path = os.path.join(OUTPUT_DIR, f"baseline_{MODEL_NAME_SAFE}_responses.jsonl")

# Resume from checkpoint
completed_ids = set()
inference_results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                completed_ids.add(rec["id"])
                inference_results.append(rec)
    print(f"Resumed: {len(completed_ids)} already completed.")

remaining = [d for d in dataset if d["id"] not in completed_ids]
print(f"Remaining: {len(remaining)}")

has_chat_template = hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None

def format_prompt(prompt_text):
    if has_chat_template:
        messages = [{"role": "user", "content": prompt_text}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"User: {prompt_text}\nAssistant:"

# --- Run inference ---
inference_start = time.time()
batch_count = 0
checkpoint_file = open(checkpoint_path, "a")

try:
    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Inference"):
        batch = remaining[i : i + BATCH_SIZE]
        prompts = [format_prompt(item["prompt"]) for item in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        for j, item in enumerate(batch):
            generated_tokens = outputs[j][input_len:]
            response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

            result = {
                "id": item["id"],
                "prompt": item["prompt"],
                "response": response,
                "constraints": item["constraints"],
                "num_constraints": item["num_constraints"],
                "difficulty_level": item["difficulty_level"],
            }
            inference_results.append(result)
            checkpoint_file.write(json.dumps(result) + "\n")

        batch_count += 1
        if batch_count % 50 == 0:
            checkpoint_file.flush()
            print(f"  Checkpoint at batch {batch_count} ({len(inference_results)} total)")
        if batch_count % 100 == 0:
            torch.cuda.empty_cache()
finally:
    checkpoint_file.close()

inference_time = time.time() - inference_start
print(f"\nInference complete: {len(inference_results)} results in {inference_time:.1f}s")
torch.cuda.empty_cache()

---
## Section 5: Evaluate & Save Results

Runs the deterministic `ConstraintChecker` on every model response and computes two metrics:

| Metric | Definition |
|--------|-----------|
| **Per-constraint CSR** | Fraction of checkable constraints passed per instance, averaged across all instances |
| **Hard CSR** | Fraction of instances where *all* checkable constraints passed |

Both metrics are computed **per difficulty level** (L1–L4) and **overall**. Results are saved as a timestamped CSV in `output/`.

In [ ]:
from evaluator import evaluate_responses, compute_metrics, save_results_csv, print_summary

eval_start = time.time()
evaluate_responses(inference_results)
metrics = compute_metrics(inference_results)
eval_time = time.time() - eval_start

total_time = inference_time + eval_time

csv_path = save_results_csv(
    results=inference_results,
    metrics=metrics,
    output_dir=OUTPUT_DIR,
    model_name=MODEL_NAME,
    label="baseline",
    elapsed_seconds=total_time,
)

print_summary(metrics, MODEL_NAME, label="baseline", elapsed_seconds=total_time)
print(f"Results saved to: {csv_path}")

---
## Section 6: Visualization

Generates three plots and saves them as PNGs to `output/`:

1. **CSR Degradation Curve** — per-constraint and hard CSR across L1 → L4 (expect a downward trend)
2. **Per-Type Bar Chart** — pass rate broken down by constraint type (e.g. `word_count`, `keyword_inclusion`, etc.)
3. **Constraint Distribution Histogram** — how many constraints per instance at each difficulty level

In [ ]:
from viz_utils import plot_csr_degradation, plot_per_type_bar, plot_constraint_distribution

plot_csr_degradation(metrics["by_level"], MODEL_NAME, OUTPUT_DIR)
plot_per_type_bar(metrics["per_type"], MODEL_NAME, OUTPUT_DIR)

dist_data = [
    {"num_constraints": r["num_constraints"], "difficulty_level": r["difficulty_level"]}
    for r in inference_results
]
plot_constraint_distribution(dist_data, MODEL_NAME, OUTPUT_DIR)
print(f"Plots saved to {OUTPUT_DIR}")

---
## Appendix: Finetuned Model Evaluation

To evaluate a finetuned model, load it the same way and call the same evaluation functions with `label="finetuned"`. The evaluator module provides `evaluate_responses()`, `compute_metrics()`, and `save_results_csv()` — just pass `label="finetuned"` to get properly labeled output files.

```python
from evaluator import evaluate_responses, compute_metrics, save_results_csv, print_summary

evaluate_responses(finetuned_results)
metrics = compute_metrics(finetuned_results)
csv_path = save_results_csv(
    results=finetuned_results,
    metrics=metrics,
    output_dir=OUTPUT_DIR,
    model_name="my-finetuned-model",
    label="finetuned",
    elapsed_seconds=total_time,
)
print_summary(metrics, "my-finetuned-model", label="finetuned", elapsed_seconds=total_time)
```